In [1]:
from mdcrow import MDCrow
from langchain.callbacks import get_openai_callback
from datetime import datetime
import os 
import traceback

In [2]:
import sys
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '../'))
sys.path.append(parent_dir)
from robustness_prompts import get_prompt # noqa: E402

prompt_3_natural = get_prompt("natural", 3)

prompt_3_natural

'Simulate 1LYZ for 1ps at 300 K. Report the secondary structure assignments of the downloaded PDB structure.'

In [3]:
llm_model = "gpt-4o-2024-08-06"
tools = "all"

In [4]:
agent = MDCrow(
    agent_type="Structured", 
    model=llm_model, 
    top_k_tools=tools, 
    use_memory=False,
    streaming=False,
    verbose=True,
)
with get_openai_callback() as cb:
    chat_start = datetime.now()
    try:
        response = agent.run(prompt_3_natural, callbacks=[cb])
        print(response)
    except Exception as e:
        exc_type, exc_value, exc_traceback = sys.exc_info()
        print(f'{type(e).__name__}:{e}')
        print("".join(traceback.format_exception(exc_type, exc_value, exc_traceback)))
    chat_end = datetime.now()
    total_runtime = (chat_end - chat_start).total_seconds()
    print(cb)
    print(f"Total runtime: {total_runtime:.2f}s")



> Entering new AgentExecutor chain...


Thought: To address the question, I need to perform two main tasks: download the PDB file for 1LYZ, set up and run a molecular dynamics simulation for 1 picosecond at 300 K, and then report the secondary structure assignments. I will start by downloading the PDB file for 1LYZ.

Action:
```
{
  "action": "PDBFileDownloader",
  "action_input": "1LYZ"
}
```PDB file found with this ID: 1LYZ

Observation: Succeeded. Downloaded the PDB file:1LYZ_214242
Thought:

To proceed, I need to set up and run a molecular dynamics simulation for the downloaded PDB file (1LYZ_214242) for 1 picosecond at 300 K. After the simulation, I will analyze the secondary structure assignments. 

Action:
```json
{
  "action": "SetUpandRunFunction",
  "action_input": {
    "pdb_id": "1LYZ_214242",
    "forcefield_files": ["amber99sb.xml", "tip3p.xml"],
    "save": true,
    "system_params": {
      "nonbondedMethod": "NoCutoff",
      "constraints": "HBonds",
      "rigidWater": true,
      "solvate": false
    },
    "integrator_params": {
      "integrator_type": "LangevinMiddle",
      "Temperature": "300 * kelvin",
      "Friction": "1.0 / picoseconds",
      "Timestep": "0.002 * picoseconds"
    },
    "simulation_params": {
      "Ensemble": "NVT",
      "Number of Steps": 500,
      "record_interval_steps": 50,
      "record_params": ["step", "potentialEnergy", "temperature"]
    }
  }
}
```error found: constraintTolerance must be specified if rigidWater is True

To correct the issue, I need to specify the `constraintTolerance` parameter since `rigidWater` is set to `True`. I will update the simulation setup with an appropriate `constraintTolerance` value and try running the simulation again.

Action:
```json
{
  "action": "SetUpandRunFunction",
  "action_input": {
    "pdb_id": "1LYZ_214242",
    "forcefield_files": ["amber99sb.xml", "tip3p.xml"],
    "save": true,
    "system_params": {
      "nonbondedMethod": "NoCutoff",
      "constraints": "HBonds",
      "rigidWater": true,
      "constraintTolerance": 0.00001,
      "solvate": false
    },
    "integrator_params": {
      "integrator_type": "LangevinMiddle",
      "Temperature": "300 * kelvin",
      "Friction": "1.0 / picoseconds",
      "Timestep": "0.002 * picoseconds"
    },
    "simulation_params": {
      "Ensemble": "NVT",
      "Number of Steps": 500,
      "record_interval_steps": 50,
      "record_params": ["step", "potentialEnergy", "temperature"]
    }
  }
}
```Building syst

About to create system...
Attempts at creating system: 0/3
adding system without solvent
Trying to add component to Forcefield...
Residue code: LYS
Residue code not in solvent list. Adding forcefield                         not supported.

Observation: Failed. No template found for residue 1 (LYS).  The set of atoms is similar to HID, but it is missing 8 atoms.  For more information, see https://github.com/openmm/openmm/wiki/Frequently-Asked-Questions#templateThis were the inputs {'pdb_id': '1LYZ_214242', 'forcefield_files': ['amber99sb.xml', 'tip3p.xml'], 'save': True, 'system_params': {'nonbondedMethod': 'NoCutoff', 'constraints': 'HBonds', 'rigidWater': True, 'constraintTolerance': 1e-05, 'solvate': False}, 'integrator_params': {'integrator_type': 'LangevinMiddle', 'Temperature': '300 * kelvin', 'Friction': '1.0 / picoseconds', 'Timestep': '0.002 * picoseconds'}, 'simulation_params': {'Ensemble': 'NVT', 'Number of Steps': 500, 'record_interval_steps': 50, 'record_params': ['step', '

...

Minimization complete!
Initial Positions saved to initial_positions.pdb
Equilibrating...


Simulating...


Done!
[2025-12-22T21:43:53.611594] Simulation ended
Total simulation time: 58.43s
Constraints must be set to 'None' for LangevinMiddle integrator.
LangevinMiddle constraints:  HBonds
Standalone simulation script written to /gpfs/fs2/scratch/qcampbe2/repos/MDCrow/ckpt/ckpt_19/simulations/NVT_1LYZ_214252_20251222_214254.py
['copy_prompt6.ipynb', 'copy_prompt8.ipynb', 'copy_prompt7.ipynb', 'TOP_sim0_214254_1LYZ_214252_20251222_214255.pdb', 'prompt1.ipynb', 'results_gpt-4o-2024-08-06', 'copy_prompt2.ipynb', '.ipynb_checkpoints', 'prompt10.ipynb', 'copy_prompt5.ipynb', 'prompt2.ipynb', 'copy_prompt3.ipynb', 'copy_prompt4.ipynb', 'copy_prompt10.ipynb', 'prompt4.ipynb', 'LOG_sim0_214254_1LYZ_214252_20251222_214255.txt', 'copy_prompt9.ipynb', 'prompt5.ipynb', 'TRAJ_sim0_214254_1LYZ_214252_20251222_214255.dcd']

Observation: Succeeded. Simulation done! 
 Summary: 
[('rec0_214255', 'Simulation trajectory for protein 1LYZ_214252 and simulation sim0_214254'), ('rec1_214255', 'Simulation state log 

To complete the task, I need to analyze the secondary structure assignments of the simulation trajectory. I will use the `ComputeDSSP` tool to determine the secondary structure for the protein trajectory.

Action:
```json
{
  "action": "ComputeDSSP",
  "action_input": {
    "traj_file": "rec0_214255",
    "top_file": "1LYZ_214252",
    "target_frames": "all"
  }
}
```
Observation: {'residues in helix': 1428, 'residues in strand': 356, 'residues in coil': 2086, 'residues not assigned, not a protein residue': 0}
Thought:

To complete the task, I have successfully simulated the protein 1LYZ for 1 picosecond at 300 K and analyzed the secondary structure assignments. Here are the results of the secondary structure analysis:

- Residues in helix: 1428
- Residues in strand: 356
- Residues in coil: 2086
- Residues not assigned (not a protein residue): 0

These results provide a detailed breakdown of the secondary structure elements present in the protein trajectory. 

Final Answer: The simulation of 1LYZ for 1 picosecond at 300 K was successful. The secondary structure analysis revealed 1428 residues in helix, 356 residues in strand, and 2086 residues in coil.

> Finished chain.
({'input': '\n    You are an expert molecular dynamics scientist, and\n    your task is to respond to the question or\n    solve the problem to the best of your ability using\n    the provided tools.\n\n    You can only respond with a single complete\n    \'Thought, Action, Action Input\' format\n    OR a single \'Final Answer\' forma

In [6]:
registry = agent.path_registry
print(registry.list_path_names_and_descriptions().replace(",", "\n"))

Files found in registry: 1LYZ_214242: PDB file downloaded from RSCB
 PDBFile ID: 1LYZ_214242
 1LYZ_214252: Cleaned File:  Removed Heterogens
 and Water Removed.  Replaced Nonstandard Residues. Added Hydrogens at pH 7.0. Missing Atoms added and nonstandard residues replaced. 
 top_sim0_214254: Initial positions for simulation sim0_214254
 sim0_214254: Basic Simulation of Protein 1LYZ_214252
 rec0_214255: Simulation trajectory for protein 1LYZ_214252 and simulation sim0_214254
 rec1_214255: Simulation state log for protein 1LYZ_214252 and simulation sim0_214254
 rec2_214255: Simulation pdb frames for protein 1LYZ_214252 and simulation sim0_214254
 rec0_214355: dssp values for trajectory with id: rec0_214255
 rec0_152216: dssp values for trajectory with id: 1LYZ_214242


In [2]:
agent=MDCrow(ckpt_dir='ckpt_19')

In [6]:
print('ckpt_dir:',os.path.basename(agent.path_registry.ckpt_dir))

ckpt_dir: ckpt_19


In [4]:
# make sure pdb was downloaded
assert os.path.exists(registry.get_mapped_path("1LYZ_214242"))

In [5]:
# make sure dssp was computed correctly
from mdcrow.tools.base_tools import ComputeDSSP

dssp = ComputeDSSP(registry)
dssp._run(traj_file= "1LYZ_214242", target_frames="first")

"{'residues in helix': 50, 'residues in strand': 14, 'residues in coil': 65, 'residues not assigned, not a protein residue': 101}"

In [8]:
# make sure trajectory and topology exist
traj_path = registry.get_mapped_path("rec0_214255")
top_path = registry.get_mapped_path("top_sim0_214254")

assert os.path.exists(traj_path)
assert os.path.exists(top_path)

In [10]:
#verify the total cost
def calculate_llm_cost(input_tokens, output_tokens, model):
    pricing_2024 = {
        "gpt-4-1106-preview": {"input": 10/1e6, "output": 30/1e6},
        "gpt-3.5-turbo-0125": {"input": 0.5/1e6, "output": 1.5/1e6},
        "gpt-4-turbo-2024-04-09": {"input": 10/1e6, "output": 30/1e6},
        "gpt-4o-2024-08-06": {"input": 5/1e6, "output": 15/1e6},
        "llama-v3p1-70b-instruct": {"input": 0.9/1e6, "output": 0.9/1e6},
        "llama-v3p1-405b-instruct": {"input": 3/1e6, "output": 3/1e6},
        "claude-3-opus": {"input": 15/1e6, "output": 75/1e6},
        "claude-3.5-sonnet": {"input": 3/1e6, "output": 15/1e6},
    }
    pricing_2025 = {
        "gpt-4-1106-preview": {"input": 10/1e6, "output": 30/1e6},
        "gpt-3.5-turbo-0125": {"input": 0.5/1e6, "output": 1.5/1e6},
        "gpt-4-turbo-2024-04-09": {"input": 10/1e6, "output": 30/1e6},
        "gpt-4o-2024-08-06": {"input": 2.5/1e6, "output": 10/1e6},
        "llama-v3p1-70b-instruct": {"input": 0.9/1e6, "output": 0.9/1e6},
        "llama-v3p1-405b-instruct": {"input": 3/1e6, "output": 3/1e6},
    }
    
    prices = pricing_2025[model]
    cost = (input_tokens * prices["input"]) + (output_tokens * prices["output"])
    return round(cost, 6)

llm_cost = calculate_llm_cost(cb.prompt_tokens, cb.completion_tokens, agent.llm.model_name)
print('Input tokens:',cb.prompt_tokens)
print('Output tokens:',cb.completion_tokens)
print('LLM costs: $',llm_cost)

Input tokens: 59027
Output tokens: 1324
LLM costs: $ 0.160808
